In [24]:
import glob
import numpy as np
import subprocess
from scipy.io import wavfile
import matplotlib.pyplot as plt
import soundfile as sf
from collections import defaultdict

In [25]:
# SETTINGS (from your log)
# =========================
SAMPLE_RATE = 384000
DTYPE = np.int16

In [26]:
# 1. Load + sort all DWV files
# =========================
files = glob.glob("*.dwv")
print(f"Found {len(files)} .dwv files")

if len(files) == 0:
    raise RuntimeError("No .dwv files found in this folder")

Found 97 .dwv files


In [27]:
groups = defaultdict(list)

for f in files:
    # Extract session key from filename:
    # e.g. 6577.230101000000.dwv → "230101"
    parts = f.split(".")
    if len(parts) < 3:
        continue

    key = parts[1][:6]  # session grouping heuristic
    groups[key].append(f)

print(f"Detected {len(groups)} session groups")

Detected 5 session groups


In [28]:
for i, (key, group_files) in enumerate(sorted(groups.items())):

    # Sort files inside each group chronologically
    group_files = sorted(group_files)

    print(f"\nProcessing Group {i+1}")
    print(f"Key: {key}")
    print(f"Files: {len(group_files)}")

    audio_chunks = []

    for f in group_files:
        try:
            data = np.fromfile(f, dtype=DTYPE)

            # Skip empty files
            if len(data) == 0:
                continue

            audio_chunks.append(data)

        except Exception as e:
            print(f"Skipping {f}: {e}")

    if len(audio_chunks) == 0:
        print("No valid audio in this group")
        continue

    # Concatenate group audio
    audio = np.concatenate(audio_chunks)

    # Normalize slightly (prevents clipping)
    audio = audio.astype(np.float32)
    audio /= np.max(np.abs(audio) + 1e-9)

    # Save full group WAV
    out_file = f"group_{i+1}_384k.wav"
    sf.write(out_file, audio, SAMPLE_RATE)

    print(f"Saved: {out_file} ({len(audio)/SAMPLE_RATE:.2f} sec)")


Processing Group 1
Key: 230101
Files: 19
Saved: group_1_384k.wav (2.04 sec)

Processing Group 2
Key: 230102
Files: 23
Saved: group_2_384k.wav (3.84 sec)

Processing Group 3
Key: 230103
Files: 18
Saved: group_3_384k.wav (0.21 sec)

Processing Group 4
Key: 230104
Files: 22
Saved: group_4_384k.wav (0.75 sec)

Processing Group 5
Key: 230105
Files: 15
Saved: group_5_384k.wav (0.39 sec)


In [29]:
print("\nDone processing all groups.")
print("Check generated group_*.wav files for full hydrophone sessions.")


Done processing all groups.
Check generated group_*.wav files for full hydrophone sessions.
